# Paper experiments: tracing mathematical error detection

This is the single Colab entry point for all three experiments. Sections 1--7 run Experiment 1: layer-wise decoding, localization, cross-domain transfer, and causal intervention. Section 9 runs the CPU-only Experiment 2 over frozen Experiment 1 outputs. Sections 10--12 run Experiment 3 transition, boundary-location, and counterfactual tests; the final sections cover optional replication, verification, and publication. The notebook is tuned for a Colab A100 with BF16 and batched model passes. Persistent status, logs, activation shards, and intervention checkpoints are written to Google Drive.

## 1. Authenticate and set up the repository

Add a fine-grained GitHub token with **Contents: read and write** permission to Colab Secrets as GITHUB_TOKEN. The token is used through a temporary askpass script and is never written to the clone URL, Git remote, notebook source, or cell output.

In [ ]:
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path


def load_github_token() -> str:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        token = getpass("GitHub token (hidden): ").strip()
    if not token:
        raise RuntimeError("A GitHub token is required to clone and publish results.")
    return token

GITHUB_TOKEN = load_github_token()
askpass_path = Path(tempfile.gettempdir()) / "math_error_github_askpass.sh"
askpass_path.write_text(
    (
        "#!/bin/sh\n"
        'case "$1" in\n'
        "*Username*) echo x-access-token ;;\n"
        '*Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    ),
    encoding="utf-8",
)
askpass_path.chmod(0o700)

def github_git_env() -> dict[str, str]:
    environment = os.environ.copy()
    environment.update({
        "GITHUB_TOKEN": GITHUB_TOKEN,
        "GIT_ASKPASS": str(askpass_path),
        "GIT_ASKPASS_REQUIRE": "force",
        "GIT_TERMINAL_PROMPT": "0",
    })
    return environment

REPOSITORY = "https://github.com/sagnikc395/tracing-mathematical-error-detection-in-language-models.git"
AUTHENTICATED_REPOSITORY = REPOSITORY.replace("https://", "https://x-access-token@", 1)
REPOSITORY_NAME = "tracing-mathematical-error-detection-in-language-models"

working_directory = Path.cwd()
if (working_directory / "pyproject.toml").exists():
    repository_directory = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_directory = working_directory.parent
else:
    repository_directory = Path("/content") / REPOSITORY_NAME
    if not (repository_directory / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", AUTHENTICATED_REPOSITORY, str(repository_directory)],
            check=True,
            env=github_git_env(),
        )

os.chdir(repository_directory)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Repository: {repository_directory}")

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
os.environ.setdefault("HF_HOME", "/content/huggingface")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab, select an A100 GPU runtime.")
gpu_name = torch.cuda.get_device_name(0)
if "A100" not in gpu_name:
    raise RuntimeError(f"This run is tuned for an A100, but Colab assigned {gpu_name}.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU/runtime does not support the required BF16 fast path.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
properties = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_name} ({properties.total_memory / 2**30:.1f} GiB)")
print("Fast path: BF16 model weights, TF32 matmul, batched extraction/interventions")

## 2. Use the fixed paper configuration

The model, dataset, split, probe, controls, bootstrap, and intervention settings come directly from `configs/experiment1.yaml`. The scientific design is unchanged. Runtime-only settings select BF16 and A100-safe batches; data, checkpoints, logs, and artifacts live on Google Drive so a Colab disconnect does not lose the run.

In [ ]:
import yaml
from google.colab import drive

drive.mount("/content/drive")
run_directory = Path("/content/drive/MyDrive/math-error-tracing")
data_path = run_directory / "data/processbench.jsonl"
artifact_directory = run_directory / "artifacts/qwen2.5-math-1.5b-a100-bf16"
data_path.parent.mkdir(parents=True, exist_ok=True)
artifact_directory.mkdir(parents=True, exist_ok=True)
log_directory = run_directory / "logs"
log_directory.mkdir(parents=True, exist_ok=True)
status_path = run_directory / "run_status.json"

with Path("configs/experiment1.yaml").open(encoding="utf-8") as config_file:
    experiment_config = yaml.safe_load(config_file)
experiment_config["data"]["output_path"] = str(data_path)
experiment_config["extraction"]["output_dir"] = str(artifact_directory)
experiment_config["model"]["dtype"] = "bfloat16"
experiment_config["extraction"]["batch_size"] = 16
experiment_config["intervention"]["batch_size"] = 8

config_path = Path("/content/paper_experiment.yaml")
config_path.write_text(yaml.safe_dump(experiment_config, sort_keys=False), encoding="utf-8")
print(config_path.read_text())

In [ ]:
import json
from datetime import datetime, timezone
from time import monotonic
from IPython.display import display


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_status(status: dict) -> None:
    temporary = status_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(status, indent=2), encoding="utf-8")
    temporary.replace(status_path)


def run_stage(stage: str) -> None:
    command = [sys.executable, "-m", "tracing_math", "--config", str(config_path), stage]
    status = json.loads(status_path.read_text()) if status_path.exists() else {"stages": {}}
    status.setdefault("stages", {})
    log_path = log_directory / f"{stage}.log"
    started_at = utc_now()
    status.update({"current_stage": stage, "status": "running", "updated_at": started_at})
    status["stages"][stage] = {"status": "running", "started_at": started_at, "log": str(log_path)}
    write_status(status)
    print(f"\n=== {stage} ===")
    print("$", " ".join(command), flush=True)
    started = monotonic()
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        log_file.write(f"\n[{started_at}] START {' '.join(command)}\n")
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        return_code = process.wait()
    elapsed = monotonic() - started
    finished_at = utc_now()
    stage_status = "complete" if return_code == 0 else "failed"
    status["stages"][stage].update(
        {"status": stage_status, "finished_at": finished_at, "elapsed_seconds": round(elapsed, 1)}
    )
    status.update({"current_stage": None, "status": stage_status, "updated_at": finished_at})
    write_status(status)
    print(f"=== {stage}: {stage_status} in {elapsed / 60:.1f} min ===")
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

run_stage("validate-config")

## 3. Download ProcessBench and extract step-boundary activations

Extraction batches 16 traces per A100 pass. Every completed shard is durable and reused after a disconnect; `extraction_progress.json`, `run_status.json`, and the live stage log show progress. Re-run this cell after reconnecting.

In [ ]:
run_stage("download-data")
run_stage("extract-activations")
extraction_progress_path = artifact_directory / "extraction_progress.json"
if extraction_progress_path.exists():
    display(json.loads(extraction_progress_path.read_text()))

## 4. Experiments A and B

Fit the layer-wise `invalid_so_far` probes, evaluate first-error localization, run the required position, lexical, shuffled-label, embedding-state, and within-error-trace controls, and compute the four-source transfer matrix. The layer and thresholds are selected on validation data; the reported metrics use the held-out test partition.

In [ ]:
run_stage("fit-probes")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

probe_directory = artifact_directory / "probes"
directions = np.load(probe_directory / "directions.npz")
selected_layer = int(directions["selected_layer"])
layer_metrics = pd.read_csv(probe_directory / "layer_metrics.csv")
controls = pd.read_csv(probe_directory / "controls.csv")
transfer = pd.read_csv(probe_directory / "domain_transfer.csv")
bootstrap_summary = pd.read_csv(probe_directory / "test_group_bootstrap_summary.csv")

print(f"Validation-selected layer: {selected_layer}")
selected_metrics = layer_metrics[
    (layer_metrics.layer == selected_layer)
    & layer_metrics.split.isin(["test", "test_error_traces"])
]
display(selected_metrics)
display(bootstrap_summary)
display(controls)
display(transfer.pivot(index="train_source", columns="test_source", values="auroc"))
display(transfer.pivot(index="train_source", columns="test_source", values="process_f1"))

## 5. Experiment C

Validate the single-token Yes/No verdict on the balanced held-out boundary sample, then run the signed dose response and matched random-orthogonal controls only when baseline specificity is nonzero. Examples are batched; results checkpoint after every learned-alpha or random-direction/alpha group. Re-running resumes from the readout-versioned checkpoint under `interventions/`.

In [ ]:
run_stage("run-interventions")
run_stage("plot")

In [ ]:
import json

intervention_directory = artifact_directory / "interventions"
behavioral_verdict = json.loads((intervention_directory / "behavioral_verdict.json").read_text())
intervention_summary = pd.read_csv(intervention_directory / "summary.csv")
effect_statistics = pd.read_csv(intervention_directory / "effect_statistics.csv")

print("Unmodified verdict metrics")
display(pd.Series(behavioral_verdict).to_frame("value"))
print("Learned-direction dose response")
display(intervention_summary[intervention_summary.direction_type == "learned"])
display(effect_statistics[effect_statistics.status == "reported"])

## 6. Confirm the paper artifacts

In [ ]:
required_artifacts = [
    "probes/layer_metrics.csv",
    "probes/test_predictions.csv",
    "probes/fit_predictions.csv",
    "probes/test_group_bootstrap_summary.csv",
    "probes/controls.csv",
    "probes/control_predictions.csv",
    "probes/domain_transfer.csv",
    "probes/directions.npz",
    "interventions/individual.csv",
    "interventions/summary.csv",
    "interventions/behavioral_verdict.json",
    "interventions/effect_statistics.csv",
    "figures/method_and_trajectory.pdf",
    "figures/predictive_results.pdf",
    "figures/transfer_and_causal.pdf",
]
artifact_check = pd.DataFrame({
    "artifact": required_artifacts,
    "exists": [(artifact_directory / path).exists() for path in required_artifacts],
})
display(artifact_check)
if not artifact_check["exists"].all():
    raise RuntimeError("The paper experiment is incomplete.")

## 7. Publish the completed paper package to GitHub

This publishes only the fixed configuration, essential result tables, learned directions, intervention results, and three paper figures. Activation shards and the dataset remain in Drive.

In [ ]:
import shutil

GITHUB_BRANCH = "main"
GIT_AUTHOR_NAME = "Colab Experiment Runner"
GIT_AUTHOR_EMAIL = "colab-experiments@users.noreply.github.com"
MAX_GITHUB_FILE_BYTES = 95 * 1024 * 1024

if not artifact_check["exists"].all():
    raise RuntimeError("The paper artifact check must pass before publishing.")

staged_before = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if staged_before:
    raise RuntimeError("The repository already has staged changes.")

publish_directory = repository_directory / "artifacts/qwen2.5-math-1.5b"
publication_files = ["extraction_identity.json", *required_artifacts]
for relative_path in publication_files:
    source = artifact_directory / relative_path
    destination = publish_directory / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
shutil.copy2(config_path, publish_directory / "experiment_config.yaml")

published_files = [
    publish_directory / relative_path for relative_path in publication_files
] + [publish_directory / "experiment_config.yaml"]
oversized = [path for path in published_files if path.stat().st_size > MAX_GITHUB_FILE_BYTES]
if oversized:
    names = ", ".join(str(path.relative_to(repository_directory)) for path in oversized)
    raise RuntimeError(f"Files exceed the safe GitHub limit: {names}")

published_paths = [str(path.relative_to(repository_directory)) for path in published_files]
subprocess.run(["git", "add", "-f", "--", *published_paths], check=True)
staged = subprocess.run(
    ["git", "diff", "--cached", "--name-only", "--", *published_paths],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if not staged:
    print("Published artifacts are already up to date.")
else:
    subprocess.run(["git", "config", "user.name", GIT_AUTHOR_NAME], check=True)
    subprocess.run(["git", "config", "user.email", GIT_AUTHOR_EMAIL], check=True)
    subprocess.run(
        ["git", "commit", "-m", "Add paper experiment artifacts", "--", *published_paths],
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--rebase", AUTHENTICATED_REPOSITORY, GITHUB_BRANCH],
        check=True,
        env=github_git_env(),
    )
    subprocess.run(
        ["git", "push", AUTHENTICATED_REPOSITORY, f"HEAD:{GITHUB_BRANCH}"],
        check=True,
        env=github_git_env(),
    )
    print(f"Published {len(published_paths)} files to {GITHUB_BRANCH}.")

## 8. Configure Experiments 2 and 3

Experiments 2 and 3 are post-hoc and do not change the frozen Experiment 1 design. Each stage has its own status key, log, artifact directory, and internal checkpoint. A completed stage is skipped after a reconnect unless `force=True` is passed to `run_checkpointed`.

In [ ]:
extended_directory = run_directory / "artifacts/experiment3-extended"
extended_directory.mkdir(parents=True, exist_ok=True)
counterfactual_pairs_path = run_directory / "data/counterfactual_pairs.jsonl"

with Path("configs/experiment3.yaml").open(encoding="utf-8") as config_file:
    extended_config = yaml.safe_load(config_file)
extended_config["experiment1_dir"] = str(artifact_directory)
extended_config["data_path"] = str(data_path)
extended_config["output_dir"] = str(extended_directory)
extended_config["model"]["dtype"] = "bfloat16"
extended_config["boundary_control"]["batch_size"] = 16
extended_config["counterfactual_patching"]["pairs_path"] = str(counterfactual_pairs_path)
extended_config["counterfactual_patching"]["batch_size"] = 8
extended_config_path = Path("/content/paper_extended_followup.yaml")
extended_config_path.write_text(
    yaml.safe_dump(extended_config, sort_keys=False), encoding="utf-8"
)

def run_checkpointed(stage_id: str, command: list[str], *, force: bool = False) -> None:
    status = json.loads(status_path.read_text()) if status_path.exists() else {"stages": {}}
    status.setdefault("stages", {})
    if not force and status["stages"].get(stage_id, {}).get("status") == "complete":
        print(f"=== {stage_id}: already complete; skipping ===")
        return
    log_path = log_directory / f"{stage_id.replace(':', '-')}.log"
    started_at = utc_now()
    status.update({"current_stage": stage_id, "status": "running", "updated_at": started_at})
    status["stages"][stage_id] = {"status": "running", "started_at": started_at, "log": str(log_path)}
    write_status(status)
    print(f"\n=== {stage_id} ===")
    print("$", " ".join(command), flush=True)
    started = monotonic()
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        return_code = process.wait()
    finished_at = utc_now()
    stage_status = "complete" if return_code == 0 else "failed"
    status["stages"][stage_id].update({
        "status": stage_status,
        "finished_at": finished_at,
        "elapsed_seconds": round(monotonic() - started, 1),
    })
    status.update({"current_stage": None, "status": stage_status, "updated_at": finished_at})
    write_status(status)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def extended_command(stage: str) -> list[str]:
    return [
        sys.executable, "-m", "tracing_math.experiment3.cli",
        "--config", str(extended_config_path), stage,
    ]

run_checkpointed("extended:validate-config", extended_command("validate-config"))

## 9. Experiment 2: CPU follow-up over frozen scores

This stage consumes the regenerated train/validation and control predictions. It runs temporal randomization, error-aligned trajectories, matched placebo onsets, within-trace discrimination, subgroup analysis, causal sensitivity, length-aware thresholds, and paired probe-control intervals. It does not load the model or activation shards.

In [ ]:
cpu_followup_directory = run_directory / "artifacts/experiment2-cpu"
cpu_followup_config = yaml.safe_load(Path("configs/experiment2.yaml").read_text())
cpu_followup_config["experiment1_dir"] = str(artifact_directory)
cpu_followup_config["data_path"] = str(data_path)
cpu_followup_config["output_dir"] = str(cpu_followup_directory)
cpu_followup_config_path = Path("/content/paper_cpu_followup.yaml")
cpu_followup_config_path.write_text(
    yaml.safe_dump(cpu_followup_config, sort_keys=False), encoding="utf-8"
)
run_checkpointed(
    "followup:cpu",
    [sys.executable, "-m", "tracing_math.followup.cli", "--config", str(cpu_followup_config_path)],
)

## 10. Experiment 3: matched onset-transition probe

The transition probe uses activation differences from the last valid boundary to the first erroneous boundary and matched transitions from correct traces. It reads Experiment 1 activation shards and writes a separate post-hoc artifact tree.

In [ ]:
run_checkpointed(
    "extended:transition-probe", extended_command("fit-transition-probe")
)

## 11. Experiment 3: natural-token versus marker-token boundary control (GPU)

One forward pass records both the final natural token of each step and the final artificial marker token. Shards are written atomically to Drive; rerunning resumes at the first missing shard. The analysis keeps Experiment 1's selected layer and regularization fixed.

In [ ]:
run_checkpointed(
    "extended:boundary-extraction", extended_command("extract-boundary-controls")
)
run_checkpointed(
    "extended:boundary-analysis", extended_command("analyze-boundary-controls")
)
display(pd.read_csv(extended_directory / "boundary_control/metrics.csv"))
display(pd.read_csv(extended_directory / "boundary_control/paired_differences.csv"))

## 12. Experiment 3: verified counterfactual activation patching (annotation gate + GPU)

The first command creates a persistent JSONL template from real first-error examples. Fill `corrected_step`, set `verified` to `true` only after checking the mathematics, and retain the original single erroneous step. The patching run is deliberately skipped until at least one verified pair exists. It checkpoints after every batch.

In [ ]:
run_checkpointed(
    "extended:counterfactual-template",
    extended_command("prepare-counterfactual-template"),
)
with counterfactual_pairs_path.open(encoding="utf-8") as handle:
    counterfactual_rows = [json.loads(line) for line in handle if line.strip()]
verified_pairs = sum(bool(row.get("verified")) for row in counterfactual_rows)
print(f"Verified counterfactual pairs: {verified_pairs}/{len(counterfactual_rows)}")
if verified_pairs:
    run_checkpointed(
        "extended:counterfactual-patching",
        extended_command("run-counterfactual-patching"),
    )
    display(pd.read_csv(extended_directory / "counterfactual_patching/summary.csv"))
else:
    print("Patching skipped. Annotate the Drive template, then rerun this cell.")

## 13. Same-family 7B replication (GPU, optional before submission)

This runs the unchanged Experiment 1 extraction and probe protocol on Qwen2.5-Math-7B-Instruct. It has an independent artifact directory and stage keys. Interventions are not run: the purpose is model-size replication, not another under-validated causal assay.

In [ ]:
RUN_7B_REPLICATION = False  # Set True only after the P0 stages are complete.
if globals().get("RUN_7B_REPLICATION", False):
    replication_directory = run_directory / "artifacts/qwen2.5-math-7b-a100-bf16"
    replication_config = yaml.safe_load(Path("configs/experiment1.yaml").read_text())
    replication_config["model"]["name"] = "Qwen/Qwen2.5-Math-7B-Instruct"
    replication_config["model"]["dtype"] = "bfloat16"
    replication_config["data"]["output_path"] = str(data_path)
    replication_config["extraction"]["output_dir"] = str(replication_directory)
    replication_config["extraction"]["batch_size"] = 8
    replication_path = Path("/content/paper_replication_7b.yaml")
    replication_path.write_text(
        yaml.safe_dump(replication_config, sort_keys=False), encoding="utf-8"
    )
    replication_base = [
        sys.executable, "-m", "tracing_math", "--config", str(replication_path)
    ]
    run_checkpointed(
        "replication-7b:extract", replication_base + ["extract-activations"]
    )
    run_checkpointed("replication-7b:fit", replication_base + ["fit-probes"])

## 14. Verify Experiment 2 and 3 artifacts

Large activation shards stay in Drive. The table distinguishes required completed analyses from annotation-gated or optional experiments.

In [ ]:
followup_checks = {
    "length-aware thresholds": cpu_followup_directory / "length_aware_threshold_results.csv",
    "paired probe-control intervals": cpu_followup_directory / "probe_control_paired_intervals.csv",
    "transition probe": extended_directory / "transition_probe/bootstrap_summary.csv",
    "boundary control": extended_directory / "boundary_control/paired_differences.csv",
    "counterfactual patching (annotation gated)": extended_directory / "counterfactual_patching/summary.csv",
}
followup_check = pd.DataFrame(
    [{"analysis": name, "path": str(path), "exists": path.exists()} for name, path in followup_checks.items()]
)
display(followup_check)
required_names = {
    "length-aware thresholds",
    "paired probe-control intervals",
    "transition probe",
    "boundary control",
}
if not followup_check[followup_check.analysis.isin(required_names)]["exists"].all():
    raise RuntimeError("One or more required follow-up stages are incomplete.")

## 15. Publish compact Experiment 2 and 3 artifacts

This copies tables, summaries, directions, and progress manifests. It excludes activation shards, intermediate checkpoints, the annotated counterfactual template, and the downloaded dataset.

In [ ]:
followup_publication_roots = [
    (cpu_followup_directory, repository_directory / "artifacts/experiment2_cpu"),
    (extended_directory, repository_directory / "artifacts/experiment3_extended"),
]
if RUN_7B_REPLICATION:
    followup_publication_roots.append((
        replication_directory / "probes",
        repository_directory / "artifacts/qwen2.5-math-7b/probes",
    ))

followup_published_files = []
for source_root, destination_root in followup_publication_roots:
    if not source_root.exists():
        continue
    for source in source_root.rglob("*"):
        if not source.is_file():
            continue
        relative = source.relative_to(source_root)
        if "shards" in relative.parts or "checkpoint" in source.name:
            continue
        destination = destination_root / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        followup_published_files.append(destination)

oversized = [
    path for path in followup_published_files if path.stat().st_size > MAX_GITHUB_FILE_BYTES
]
if oversized:
    names = ", ".join(str(path.relative_to(repository_directory)) for path in oversized)
    raise RuntimeError(f"Follow-up files exceed the safe GitHub limit: {names}")
followup_paths = [
    str(path.relative_to(repository_directory)) for path in followup_published_files
]
if followup_paths:
    subprocess.run(["git", "add", "-f", "--", *followup_paths], check=True)
    staged = subprocess.run(
        ["git", "diff", "--cached", "--name-only", "--", *followup_paths],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if staged:
        subprocess.run(["git", "config", "user.name", GIT_AUTHOR_NAME], check=True)
        subprocess.run(["git", "config", "user.email", GIT_AUTHOR_EMAIL], check=True)
        subprocess.run(
            ["git", "commit", "-m", "Add workshop follow-up artifacts", "--", *followup_paths],
            check=True,
        )
        subprocess.run(
            ["git", "pull", "--rebase", AUTHENTICATED_REPOSITORY, GITHUB_BRANCH],
            check=True, env=github_git_env(),
        )
        subprocess.run(
            ["git", "push", AUTHENTICATED_REPOSITORY, f"HEAD:{GITHUB_BRANCH}"],
            check=True, env=github_git_env(),
        )
    else:
        print("Compact follow-up artifacts are already up to date.")